# 00 · Setup
Crea el catálogo del proyecto, un esquema por capa y los volumes de landing y checkpoints. Se corre una sola vez.

In [0]:
%run ./00_config

In [0]:
def show(df, n=50):
    print(df.limit(n).toPandas().to_string(index=False))

if not schema:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
    for s in ["landing", "bronze", "silver", "gold", "ops"]:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{s}")
    for v in ["landing.raw_zips", "landing.csv", "landing.ref", "ops.checkpoints"]:
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{v}")
    show(spark.sql(f"SHOW SCHEMAS IN {catalog}"))
else:
    print(f"modo esquema único: se usa {catalog}.{schema} y el volume {volume} (no se crean esquemas)")

In [0]:
for t in ["silver.ais_posiciones", "gold.trafico_celda_hora", "gold.trayectorias_diarias", "gold.presencia_puertos"]:
    n = spark.table(f"oceanwatch_e2.{t}").filter("fecha = DATE'2023-06-08'").count()
    print(f"{t:28s} {n:>12,}")
spark.table("oceanwatch_e2.ops.quality_gate_log").orderBy("ts_evaluacion").show(truncate=False)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T("gold", "dim_tipo_buque")} AS
SELECT * FROM VALUES
  (30, 30, 'Pesca'), (31, 32, 'Remolque'), (33, 34, 'Dragado / buceo'), (35, 35, 'Militar'),
  (36, 36, 'Vela'), (37, 37, 'Recreo'), (40, 49, 'Alta velocidad'), (50, 59, 'Servicio (práctico, SAR, remolcador)'),
  (60, 69, 'Pasajeros'), (70, 79, 'Carga'), (80, 89, 'Tanquero'), (90, 99, 'Otro')
AS t(codigo_min, codigo_max, categoria)
""")
print(T("gold", "dim_tipo_buque"), "→", spark.table(T("gold", "dim_tipo_buque")).count(), "categorías")